In [ ]:
PREFIX='https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main'

!wget ${PREFIX}/01-agentic-rag/code/ingest.py
!wget ${PREFIX}/01-agentic-rag/code/rag_helper.py
!wget ${PREFIX}/04-evaluation/code/evaluation_utils.py  

://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py: Scheme missing.
://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py: Scheme missing.
://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py: Scheme missing.


In [5]:
from ingest import load_faq_data
documents = load_faq_data()

In [6]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [7]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [8]:
documents = documents_llm

In [9]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [10]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [11]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [13]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [14]:
import json

user_prompt = json.dumps(doc)

In [15]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [16]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [17]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [21]:
response.output_parsed.questions

['I just found this course — can I still join now, or is it too late?',
 'Am I allowed to start the course after it has already begun?',
 'If I join late, will I still be able to get a certificate?',
 'What do I need to do to qualify for the certificate if I’m joining now?',
 'Is there still time to submit the project and get certified?']

In [22]:
from evaluation_utils import llm_structured
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — can I still join it now?', 'Is it too late to start the course if I discovered it after it began?', 'Can I enroll in the course even though it has already started?', 'If I join late, am I still eligible for a certificate?', 'What do I need to do to get the certificate if I’m joining the course now?']


In [23]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=89, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=296)

In [24]:
from evaluation_utils import calc_price

In [25]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.0004005, 'total_cost': 0.00055575}

In [26]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — can I still join it now?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course if I discovered it after it began?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll in the course even though it has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to get the certificate if I’m joining the course now?',
  'document': '74eb249bbf'}]

In [27]:
import pandas as pd
pd.DataFrame(records)

,question,document
0,I just found this course — can I still join it...,74eb249bbf
1,Is it too late to start the course if I discov...,74eb249bbf
2,Can I enroll in the course even though it has ...,74eb249bbf
3,"If I join late, am I still eligible for a cert...",74eb249bbf
4,What do I need to do to get the certificate if...,74eb249bbf


In [28]:
from evaluation_utils import llm_structured_retry


In [30]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [31]:
generate_ground_truth(doc)

([{'question': 'Can I still join the course if I found it late, or is it already closed?',
   'document': '74eb249bbf'},
  {'question': 'If I join now, am I still eligible for a certificate?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to get the certificate after joining late?',
   'document': '74eb249bbf'},
  {'question': 'Is it okay to start the course after it has already begun?',
   'document': '74eb249bbf'},
  {'question': 'Do I have to submit the project before submissions close to receive the certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=89, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=296))

In [32]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [33]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [34]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [35]:
results[1]

([{'question': 'I signed up for LLM Zoomcamp, but I still haven’t gotten any confirmation email — is that normal?',
   'document': '977bf7786c'},
  {'question': 'Do I need to wait for an acceptance email before I can start the LLM Zoomcamp materials and homework?',
   'document': '977bf7786c'},
  {'question': 'If I registered for the course, does that mean I’m officially in, or is there another approval step?',
   'document': '977bf7786c'},
  {'question': 'Can I begin watching the LLM Zoomcamp lessons and submitting assignments even if I never received any email?',
   'document': '977bf7786c'},
  {'question': 'What’s the point of the registration form for LLM Zoomcamp if it doesn’t affect whether I can take the course?',
   'document': '977bf7786c'}],
 ResponseUsage(input_tokens=238, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=129, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=367))

In [36]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [37]:
ground_truth[0]

{'question': 'Is it still possible to join the course if I found out about it late?',
 'document': '74eb249bbf'}

In [38]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07758524999999998

In [39]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07758524999999998

In [40]:
df_ground_truth = pd.DataFrame(ground_truth)

In [41]:
df_ground_truth

,question,document
0,Is it still possible to join the course if I f...,74eb249bbf
1,Can I start the course now even if I missed th...,74eb249bbf
2,"If I join the course late, can I still get a c...",74eb249bbf
3,What do I need to do to earn the certificate i...,74eb249bbf
4,Does late enrollment affect my ability to subm...,74eb249bbf
...,...,...
510,How do I get pip to install requests 2.32.3 in...,4b30b918bc
511,What command can I use to install requests str...,4b30b918bc
512,Why am I seeing a 401 Client Error while setti...,4b30b918bc
513,Is there a way to force the requests package t...,4b30b918bc


In [45]:
import os

os.makedirs("data", exist_ok=True)
df_ground_truth.to_csv("data/ground_truth.csv", index=False)